This notebook shows why rendering meshes and Gaussians **together** is useful: a **coarse textured mesh plus a few Gaussians** reproduces a detailed object better than either of them alone.

**Ground truth.** The detailed textured Spot mesh (5856 triangles), lit by a point light, pre-rendered from 24 orbit views (training) and 6 other views (testing).

**Models**, trained on the same images (Adam, L1 loss):

1. **Mesh only**: a coarse proxy mesh (360 triangles: the Spot control mesh with its vertices snapped onto the detailed surface), with a learned texture map. Its geometry is fixed, like a template mesh (pass `train_vertices=True` to `train_model` to learn it too).
2. **Gaussians only**: K 3D Gaussians (plain 3DGS), initialized on the coarse mesh.
3. **Hybrid**: the coarse textured mesh **and** K Gaussians, rendered together with Gmesh and trained jointly. The mesh carries the large textured surfaces; the Gaussians add the volume and detail the flat triangles of the coarse mesh miss (round shapes, ears, horns, legs), in front of its surface. Gaussians behind the mesh are hidden, Gaussians crossing it are cut smoothly, and the gradients reach both.

The helpers are shared with the script version, `examples/train_hybrid_toy.py`.

In [ ]:
## Computing Device
device = 'cuda:0'
import torch
torch.cuda.set_device(device)

import os, sys
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

sys.path.insert(0, os.path.abspath('..'))  # the repository root (this notebook runs from ./examples)
sys.path.insert(0, os.path.abspath('.'))   # ./examples, for train_hybrid_toy.py

from gmesh.scenes.cameras import OrbitCamera
from train_textured_toy import load_scene, orbit_poses, make_gaussians
from train_hybrid_toy import load_coarse_mesh, make_mesh, initial_mesh_params, make_renderers, train_model

torch.manual_seed(0)

### Ground truth and the coarse mesh

In [ ]:
IMAGE_SIZE = 256
ITERS = 3000        # training iterations per model
N_GAUSSIANS = 2000  # Gaussian budget of the Gaussians-only and hybrid models

gt_mesh, light = load_scene(device)       # detailed textured Spot mesh and a point light
coarse = load_coarse_mesh(device, snap_to=gt_mesh.verts)  # (verts, faces, uvs, faces_uv) of the coarse proxy mesh
print(f'ground truth: {gt_mesh.num_faces} triangles; coarse mesh: {coarse[1].shape[0]} triangles')

camera = OrbitCamera(image_width=IMAGE_SIZE, image_height=IMAGE_SIZE, fov_x=40, fov_y=40,
                     z_near=0.1, z_far=100, device=device)
renderers = make_renderers(camera, light)   # MeshRenderer, GaussianRasterizer, GmeshRenderer (same light)
mesh_renderer, gaussian_rasterizer, gmesh_renderer = renderers
train_poses, test_poses = orbit_poses(device)

with torch.no_grad():
    gt_train = [mesh_renderer.render(gt_mesh, pose)[0][0] for pose in train_poses]
    gt_test = [mesh_renderer.render(gt_mesh, pose)[0][0] for pose in test_poses]

def to_np(img):
    return img.detach().clamp(0, 1).cpu().numpy()

def show_images(images, titles, size=3.0):
    fig, axs = plt.subplots(1, len(images), figsize=(size * len(images), size))
    for ax, img, title in zip(np.atleast_1d(axs), images, titles):
        ax.imshow(to_np(img))
        ax.set_title(title, fontsize=10)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

# the coarse mesh, untrained (gray texture)
with torch.no_grad():
    coarse_views = [mesh_renderer.render(make_mesh(coarse, initial_mesh_params(coarse, 256)), pose)[0][0]
                    for pose in test_poses[:3]]
show_images(gt_test[:3] + coarse_views, ['ground truth'] * 3 + ['coarse mesh (untrained)'] * 3, size=2.6)

### Train the three models

Each model is trained for the same number of iterations. `train_model` returns the training history (losses, test PSNR, test-view snapshots, number of parameters) and the trained parameters.

In [ ]:
results = {}
for model in ['mesh', 'gaussians', 'hybrid']:
    print(f'=== {model} ===')
    hist, m_params, g_params = train_model(model, renderers, coarse, N_GAUSSIANS, gt_train, gt_test,
                                           train_poses, test_poses, ITERS, snapshot_every=ITERS // 6,
                                           log_every=ITERS // 6)
    results[model] = (hist, m_params, g_params)
    print(f"test PSNR {hist['psnr'][0]:.2f} -> {hist['psnr'][-1]:.2f} dB, {hist['n_params']} parameters\n")

fig, axs = plt.subplots(1, 2, figsize=(10, 3.4))
for model, (hist, _, _) in results.items():
    axs[0].plot(hist['loss'], linewidth=0.6, label=model)
    axs[1].plot(hist['psnr_iters'], hist['psnr'], 'o-', label=model)
axs[0].set_yscale('log'); axs[0].set_title('training L1 loss'); axs[0].set_xlabel('iteration'); axs[0].legend()
axs[1].set_title('test PSNR (dB)'); axs[1].set_xlabel('iteration'); axs[1].legend()
plt.tight_layout(); plt.show()

### Compare on the test views

In [ ]:
for view in range(3):
    show_images([gt_test[view]] + [results[m][0]['snapshots'][-1][1][view] for m in ['mesh', 'gaussians', 'hybrid']],
                ['ground truth', 'coarse mesh only', f'{N_GAUSSIANS} Gaussians only', f'hybrid: coarse mesh + {N_GAUSSIANS} Gaussians'])

### What does each part of the hybrid model learn?

The trained hybrid model rendered as its mesh alone, its Gaussians alone (no occlusion), and both together (Gmesh): the mesh carries the body and its texture, the Gaussians fill in what the coarse mesh misses.

In [ ]:
_, m_params, g_params = results['hybrid']
with torch.no_grad():
    for pose in test_poses[:3]:
        ret = gmesh_renderer.render(mesh=make_mesh(coarse, m_params), gaussians=make_gaussians(g_params),
                                    camera_pose=pose, return_all=True)
        show_images([ret['mesh'][0][0], ret['3dgs'][0][0], ret['blend'][0][0]],
                    ['its mesh', 'its Gaussians', 'both (Gmesh)'])
show_images([torch.sigmoid(m_params['texture_logits'])], ['learned texture of the coarse mesh'], size=4.0)

### Gaussian budget sweep (optional, slower)

Test PSNR of the Gaussians-only and hybrid models for several numbers of Gaussians, with the coarse mesh alone as a reference.

In [ ]:
BUDGETS = [500, 2000, 8000]
sweep = {}
for k in BUDGETS:
    for model in ['gaussians', 'hybrid']:
        hist = results[model][0] if k == N_GAUSSIANS else \
            train_model(model, renderers, coarse, k, gt_train, gt_test, train_poses, test_poses, ITERS, log_every=0)[0]
        sweep[(model, k)] = hist['psnr'][-1]
        print(f'{model:10s} {k:6d} Gaussians: test PSNR {sweep[(model, k)]:.2f} dB')

plt.figure(figsize=(5.5, 4))
plt.axhline(results['mesh'][0]['psnr'][-1], color='gray', linestyle='--', label='coarse mesh only')
plt.plot(BUDGETS, [sweep[('gaussians', k)] for k in BUDGETS], 'o-', label='Gaussians only')
plt.plot(BUDGETS, [sweep[('hybrid', k)] for k in BUDGETS], 'o-', label='hybrid (coarse mesh + Gaussians)')
plt.xscale('log'); plt.xlabel('number of Gaussians'); plt.ylabel('test PSNR (dB)'); plt.legend()
plt.tight_layout(); plt.show()

### Summary

In [ ]:
for model in ['mesh', 'gaussians', 'hybrid']:
    hist = results[model][0]
    print(f"{model:10s} test PSNR {hist['psnr'][-1]:6.2f} dB   ({hist['n_params']} parameters)")